# Artificial Intelligence - Lab Project - Part 2

## Building a "Traditional" RAG system and evaluating it

**Names:**   Αικατερίνα Γεωγράνα
**Student IDs:**   ge20097
**Team ID:**    Ομάδα 28

This project has two steps:
One is about designing a traditional RAG system on the movie script "12 Angry Men". The chunk retrieval is based on a combination of keyword and vector search.

The second is about evaluating your system based on a dataset which will be common with all other teams and is made up of a few questions you provided in Part 1 of the Assignment.

## Deliverables:
You will be designing a basic RAG system in this notebook. The necessary python libraries will be given to you for each task but you will have to write the actual code. Specific instructions will be given above each cell.
You only need to implement the code that will be instructed in comments in the code in the notebook's cells.

## What you need to upload
### A zip file named: ``` AI_Assignment_2_Team_{REPLACE WITH TEAMID}.zip" ``` containing:

- This notebook completed and with the cell outputs (named: `AI_Assignment_2_Team_{REPLACE WITH TEAMID}_notebook.ipynb` )
- The csv evaluation file (named: `AI_Assignment_2_Team_{REPLACE WITH TEAMID}_evaluation.csv`

## Please be careful with file naming and format!


# Visualize the dataset

To start with, upload the dataset in Google Colab. Then you can print the first lines to make sure it is correct.

In [16]:
import pandas as pd

# Load the dataset
df = pd.read_csv("dataset.csv")

display(df.head())

,Question,Correct Answer,Page References,Explanation
0,What type of train passes by the apartment rig...,An el train (elevated train).,26,This requires retrieving a specific environmen...
1,"According to the boy's alibi, where did he cla...",He claimed to be at the movies.,25,Retrieval of a specific plot point established...
2,What does the sign on the first door of the co...,Court of General Sessions. Part I,3,Fact explicitly stated in the script in the de...
3,What pattern does Juror #3 draw on the same sh...,tic-tac-toe pattern,64,Fact explicitly stated in the script. The info...
4,Which Juror says that he's lived in a slum all...,Juror #5,31,Fact explicitly stated in the script on page 31


# Building a Traditional RAG System: 12 Angry Men

In this exercise, we are going to build a basic Retrieval-Augmented Generation (RAG) system from scratch. Our target data is the script for the movie *12 Angry Men*.

A Large Language Model (LLM) alone might not know specific details about this script. To fix this, we will:
1. Download the raw PDF of the script and extract its text.
2. Break the script down into smaller pieces (chunks).
3. Store these chunks in Keyword (BM25) and Vector (FAISS) databases.
4. Search both databases when a question is asked.
5. Provide the retrieved chunks to the LLM so it can answer the question accurately.

# Step 1: Install required libraries
In this first step, we install all the necessary Python packages for building our retrieval and language model pipeline.

The command below uses `pip` to quietly (`-q`) install:

- **transformers** → for loading and using large language models (LLMs)
- **accelerate** → for efficient model execution (especially on GPUs)
- **sentence-transformers** → for generating text embeddings
- **faiss-cpu** → for fast similarity search in vector databases
- **rank_bm25** → for traditional keyword-based retrieval
- **pandas** → for handling and analyzing tabular data
- **PyMuPDF** → for reading and processing PDF documents

> Run this cell once before continuing with the rest of the notebook.


In [17]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu rank_bm25 pandas PyMuPDF

## Step 2: Data Ingestion & Document Chunking

Before we can search through our movie script, we need to load the PDF and break it down into smaller, digestible pieces called **chunks**.

Why do we chunk?
1. **Context Limits:** Large Language Models (LLMs) have a maximum context window. We can't feed a 100-page PDF into the prompt all at once.
2. **Retrieval Accuracy:** If our search system returns massive blocks of text, the LLM might get distracted by irrelevant information ("lost in the middle" effect). Smaller, focused chunks lead to more accurate answers. How many chunks we provide the LLM is not set of course, depending on the LLM capabilities we can give smaller or larger number of chunks.


### Chunking for this assignment: Structural Chunking (Page-Based)
Instead of arbitrary word counts, we could use the document's natural structure. Of course there are other ways to create the chunks as there is no requirement for them to be of equal size.

In the cell below, we will:
1. Download the script for *12 Angry Men*.
2. Iterate through the PDF **page-by-page** using `PyMuPDF`.
3. Save each page as its own chunk.

> **Tip for RAG:** Notice how we inject `--- Page X ---` at the top of every chunk. This is a form of **metadata enrichment**. By embedding the page number directly into the text, we give the LLM the ability to cite its sources when answering the user!

In [18]:
import requests
import fitz  # PyMuPDF

# 1. Download the movie script PDF
url = "https://f004.backblazeb2.com/file/screenplays/posts/12-angry-men-1957/scripts/12%20Angry%20Men%20-%20Release.pdf"
pdf_path = "12_angry_men.pdf"

print("Downloading the script PDF...")
response = requests.get(url)
with open(pdf_path, "wb") as f:
    f.write(response.content)
print("Download complete!")

# 2. Extract text and chunk by page
print("Extracting text from PDF by page...")
doc = fitz.open(pdf_path)

chunks = []

for page_num, page in enumerate(doc):
    page_text = page.get_text().strip()

    # We only add the page if it actually contains text (skips blank pages)
    if page_text:
        # Pro-tip: Adding the page number to the chunk helps the LLM cite its sources!
        chunk_content = f"--- Page {page_num + 1} ---\n{page_text}"
        chunks.append(chunk_content)

print(f"Created {len(chunks)} chunks from {len(doc)} pages.")

# Let's peek at an example chunk
print("\n--- Example Chunk [10] (Page 11) Preview ---")
print(chunks[10][:500]) # Printing the first 500 characters of the 11th chunk
print("--------------------------------------------")

Download complete!
Extracting text from PDF by page...
Created 149 chunks from 149 pages.

--- Example Chunk [10] (Page 11) Preview ---
--- Page 11 ---
11.
#3
(smiling)
I didn't get a chance to look at the
papers today.
#4
I was just wondering how the market
closed.
#3
(pleasantly)
I wouldn't knew. Say, are you on the
exchange or something.
#4
I'm a broker.
#3
Well that's very interesting.
Listen, maybe you can answer a
question for me. I have an uncle
who’s been playing around with some
Canadian stuff...
The foreman turns around, and, as if it is an effort, calls 
out loudly to the others.
FOREMAN
All right, gentlemen. Let’s ta
--------------------------------------------


## Step 4: Building the Retrieval Databases

To enable accurate information retrieval within our RAG system, we need efficient methods to parse and search our text chunks. In this section, you will set up the core components of a hybrid search system by implementing two distinct search strategies: a sparse keyword database and a dense vector database.

*(Note: Assume the variable `chunks`, a list of text strings, is already available in your environment from the previous steps.)*

### 1. Keyword Search (BM25): Lexical Matching
BM25 operates as an advanced algorithmic version of a traditional keyword index.
* **Mechanism:** It identifies exact word matches between the user's query and the text chunks using a statistical scoring framework (based on Term Frequency-Inverse Document Frequency). It heavily weights rare, highly specific terms (e.g., "photosynthesis") while discounting common vocabulary (e.g., "the," "and").
* **Strengths:** Highly effective for retrieving specific entities, proper names, exact quotes, or strict technical jargon.
* **Limitations:** Relies strictly on lexical matching. It will fail to retrieve a chunk that exclusively uses a synonym (e.g., a query for "automobile" will not match a chunk only containing "car").

### 2. Vector Search (FAISS): Semantic Matching
Vector search identifies relevant information based on underlying meaning and context rather than exact phrasing.
* **Mechanism:** Using an embedding model, we translate text chunks into high-dimensional numerical vectors (embeddings). Texts with similar semantic meanings are positioned closer together in this mathematical space. FAISS (Facebook AI Similarity Search) rapidly calculates the spatial distance between these vectors.
* **Strengths:** Excels at understanding context and conceptual relationships (e.g., a query for "canines playing" will successfully retrieve a text chunk about "dogs fetching a ball").
* **Limitations:** Because it evaluates broad meaning, it can be overly generalized and underperform when a query requires a rigid, exact match (such as a specific user ID).

---

### Your Tasks:

**1. Keyword Database (BM25)**
* Tokenize the text `chunks` by splitting them by simple spaces.
* Initialize a `BM25Okapi` object with these tokenized chunks.

**2. Vector Database (FAISS)**
* Load the `'all-MiniLM-L6-v2'` embedding model using `SentenceTransformer`.
* Use this model to encode our `chunks` into vector embeddings.
* Find the dimensionality of these newly created embeddings.
* Create a `faiss.IndexFlatL2` index using that exact dimension.
* Add your embeddings to the FAISS index (remember to convert them to a NumPy array if they aren't already).

In [19]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("Building Keyword Database (BM25)...")

# TODO: Tokenize 'chunks' (simple space splitting)
tokenized_chunks = [chunk.split() for chunk in chunks]

# TODO: Initialize the BM25Okapi object
bm25 = BM25Okapi(tokenized_chunks)

print("Building Vector Database (FAISS)...")

# TODO: Load the 'all-MiniLM-L6-v2' embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# TODO: Convert the text chunks into vector embeddings
embeddings = embedder.encode(chunks)

# TODO: Get the dimension of the embeddings
dimension = embeddings.shape[1]

# TODO: Create a FAISS IndexFlatL2 index using the dimension ("IndexFlatL2" function)
vector_db =  faiss.IndexFlatL2(dimension)

# TODO: Add the numpy array of embeddings to the FAISS index
vector_db.add(np.array(embeddings))

Building Keyword Database (BM25)...
Building Vector Database (FAISS)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step 5: The Hybrid Retriever

Now it is time to build the engine of our RAG system. You will write a function that takes a user's query, searches *both* of our databases, and returns the most relevant context.

Because BM25 (lexical) and FAISS (semantic) excel at different types of matching, using them together yields much better results than using either alone.

**Your Task:**
Create a `hybrid_retrieve` function that executes the following pipeline:
1. **Keyword Search:** Tokenize the incoming query, score it against your `bm25` database, and extract the indices of the top-scoring chunks.
2. **Vector Search:** Encode the query using your `embedder`, search the `vector_db`, and extract the indices of the closest vectors.
3. **The Fusion Challenge (Your Choice):** You now have two lists of "top indices." You must design a way to combine them into a single, final list of indices.
    * *Basic Approach:* Combine the lists and use a Python `set` to remove duplicates.
    * *Advanced Approach (Optional):* Implement a scoring algorithm like **Reciprocal Rank Fusion (RRF)** to mathematically weigh and re-rank the indices based on their positions in both original lists.
4. **Fetch Context:** Map your final combined indices back to the original `chunks` list to get the actual text.

### Expected Outcome
Your function should return a Python **list of text strings** (the actual text chunks). When you run the test query at the bottom of the cell, it should print the relevant text chunks that best answer the question.

In [20]:
def hybrid_retrieve(query, top_k=2):
    """
    Retrieves top chunks using BM25 and Vector Search, then combines the results.
    """

    # --- 1. Keyword Search (BM25) ---
    # TODO: Tokenize the query (split by spaces)
    tokenized_query = query.split()

    # TODO: Get scores for the query from the 'bm25' object ("get_scores" function)
    bm25_scores = bm25.get_scores(tokenized_query)

    # TODO: Get the indices of the top-scoring chunks
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:top_k * 5]


    # --- 2. Vector Search (FAISS) ---
    # TODO: Encode the query into a vector using your 'embedder'
    query_embedding = embedder.encode(query)


    # TODO: Search the 'vector_db' to get the top indices ("search" function )
    # Fetch more top initially to allow for better fusion and deduplication
    distances, faiss_top_indices = vector_db.search(np.array([query_embedding]).astype('float32'), top_k * 5)
    # Hint: FAISS returns a 2D array, you may need to extract the first element (e.g., [0].tolist())
    faiss_top_indices = faiss_top_indices[0].tolist()

    # --- 3. Combine ---
     # TODO: Implement your strategy to combine 'bm25_top_indices' and 'faiss_top_indices'.
    # You can use simple deduplication (sets), alternating selection, or advanced algorithms like RRF.
    # Ensure your final list is no longer than the desired output size (though you may fetch more initially).
    # Implement the basic fusion strategy: Combine lists and use a set to remove duplicates
    combined_indices_set = set(bm25_top_indices).union(set(faiss_top_indices))

    # Convert back to a list and ensure indices are valid within the range of 'chunks'
    # Sort to maintain some consistency, though the order might not be strictly ranked after set operation
    valid_combined_indices = sorted([idx for idx in list(combined_indices_set) if 0 <= idx < len(chunks)])

    # Limit to the desired top_k
    final_combined_indices = valid_combined_indices[:top_k]

    # --- 4. Fetch the Text ---
    # TODO: Fetch the actual text strings from the original 'chunks' list using your final indices
    retrieved_chunks = [chunks[idx] for idx in final_combined_indices]

    return retrieved_chunks


# ==========================================
# You can test your hybrid retrieval function with a sample query. Adjust the query and top_k as needed to see how your fusion strategy performs.
# ==========================================
test_query = "Who had a cold?"

# Feel free to adjust top_k to see how your fusion strategy handles more chunks
results = hybrid_retrieve(test_query, top_k=2)

print(f"Found {len(results)} relevant chunks for the test query.\n")
for i, chunk in enumerate(results):
    print(f"--- Chunk {i + 1} ---")
    print(chunk)
    print()

Found 2 relevant chunks for the test query.

--- Chunk 1 ---
--- Page 2 ---
2.
Juror #9: 70 years old. Retired. A mild, gentle old man, long 
since defeated by life, and now merely waiting to die. A man 
who recognizes himself for what he is, and mourns the days
when it would have been possible to be courageous without 
shielding himself behind his many years. From the way he take 
pills whenever he is excited, it is obvious that he has a 
heart condition.
Juror #10: 46 years old. Garage owner. An angry, bitter man.
A man who antagonizes almost at sight. A bigot who places no 
values on any human life save his own. A man who has been no 
where and is going nowhere and knows it deep within him. He 
has a bad cold and continually blows his nose, sniffs a ben-
zedrine inhaler, etc.
Juror #11: 48 years old. Watchmaker. A refugee from Europe 
who has come to this country in 1941. A man who speaks with 
an accent and who is ashamed, humble, almost subservient to 
the people around him, but a

## Step 6: Setting up the LLM
We will load `Qwen3-4B-Instruct-2507`. We load it in `bfloat16` precision to save memory so it fits comfortably on our free Colab GPU.

In [21]:
import torch
from transformers import pipeline

print("Loading LLM...")

model_id = "Qwen/Qwen3-4B-Instruct-2507"

# Set up the text-generation pipeline
llm_pipeline = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"dtype": torch.bfloat16},
    device_map="auto"
)

print("LLM Loaded and ready!")

Loading LLM...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

LLM Loaded and ready!


## Step 7: The Final RAG Pipeline

You have built the retrieval engine and loaded the generator. Now, it is time to tie everything together into a complete Retrieval-Augmented Generation (RAG) pipeline.

In this final step, you will write a function that takes a user's question, fetches the most relevant information from your databases, constructs a prompt for the LLM, and generates a final answer.

**Your Tasks:**
1. **Retrieve Context:** Call your hybrid retrieval function from Step 5 to get the top `k` relevant text chunks for the user's query. Join these chunks into a single, cohesive string.
2. **Design the Prompt:** Large Language Models perform best with clear instructions. You need to construct a message history (a list of dictionaries) containing:
   * A **System Prompt:** Define the AI's persona (e.g., an expert analyzing the movie "12 Angry Men"). Crucially, instruct the AI to answer *only* using the provided context and to cite its sources if possible.
   * A **User Prompt:** Combine the user's actual question with the context string you built in step 1.
3. **Generate Answer:** Pass your message history into your `llm_pipeline`.
   * *Hint:* To keep the model grounded and prevent rambling, configure parameters like `max_new_tokens=150`, `temperature=0.1` (low temperature reduces hallucinations), and `do_sample=True`.
4. **Extract Response:** The Hugging Face pipeline returns a nested structure. You will need to extract just the text of the assistant's final generated reply.

### Expected Outcome
Your function should return a single, coherent text string containing the AI's answer. When you run the test queries at the bottom of the cell, the LLM should accurately answer the questions based exclusively on the retrieved script excerpts.

In [26]:
def ask_rag(query):
    """
    Executes the full RAG pipeline: Retrieves context, builds prompt, and generates an answer.
    """
    print(f"Question: {query}")
    print("Retrieving context...")
    contexts = hybrid_retrieve(query, top_k=3)

    # Join the retrieved chunks into a single string (e.g., separated by newlines)
    context_string = "\n\n---\n\n".join(contexts)

    # 2. Build the prompt
    # Write a strong system prompt directing the AI's behavior
    system_prompt = """You are an expert assistant analyzing the movie "12 Angry Men". You should answer all the questions based exclusively on the provided context and cite your sources. Do not invent facts or adhere to specific formatting requirements. Communicate friendly and formaly.
"""
    # Combine the context and the user's query / Change this as needed to fit your system prompt and the way you want to present the information to the LLM.
    user_prompt = f"Context :\n{context_string}\n\nQuestion: {query}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # 3. Generate Answer / You can play with the generation parameters to see how they affect the output. For factual answers, lower temperature and disabling sampling can help.
    outputs = llm_pipeline(
        messages,
        max_new_tokens=150,
        temperature=0.1,
        do_sample=False
    )

    # Extract just the text of the assistant's reply
    final_answer = outputs[0]["generated_text"][-1]["content"]

    return final_answer

## Step 8: Baseline Comparison (LLM Without RAG)

To truly understand the value of our RAG pipeline, we need to see how the Large Language Model performs *without* it.

When you ask a standard LLM a highly specific question, it relies entirely on the knowledge it memorized during its initial training. If it doesn't know the answer, it might guess, hallucinate, or give a vague response.

**Your Task:**
Simply run the cell below. We have provided the `ask_baseline` function for you. This function asks the exact same LLM your question, but it **does not** provide any of the script excerpts from our database.

After running this, try asking both `ask_rag(query)` and `ask_baseline(query)` the same highly specific questions about the script and compare the results!

In [27]:
def ask_baseline(query):
    """
    Queries the LLM directly without any retrieved context (Baseline).
    """
    print(f"Question: {query}")
    print("Generating baseline answer...")

    # 1. Build the prompt without context
    system_prompt = "You are a helpful assistant. Answer the user's question to the best of your ability. The question will be about the movie '12 Angry men' of 1957."

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ]

    # 2. Generate Answer
    outputs = llm_pipeline(
        messages,
        max_new_tokens=150,
        temperature=0.1,
        do_sample=False
    )

    # 3. Extract just the text of the assistant's reply
    final_answer = outputs[0]["generated_text"][-1]["content"]
    return final_answer

## Step 9: Evaluating Our System

A crucial part of building AI systems is evaluating them. To do this, we will run both our Baseline LLM and our newly built RAG pipeline against a dataset of predefined questions about "12 Angry Men."

We will compare their answers to the `Correct Answer` provided in our dataset to see how much the retrieved context improves the model's accuracy.

**Your Task:**
Run the evaluation code block below.
* Set `PRINT_RESULTS = True` if you want to watch the models answer the questions in real-time.
* Set `PRINT_RESULTS = False` if you just want it to process everything silently in the background.

The script will automatically test the first question - you can change that variable as you wish. Once it finishes, it will save a new file called `dataset_with_results.csv` containing all the generated answers.

In [28]:
import pandas as pd
from IPython.display import display, Markdown

# ==========================================
# EVALUATION SETTINGS
# ==========================================
# Set this to False if you only want to generate the CSV without printing to the screen
PRINT_RESULTS = True

# Define how many questions to test (e.g., first 5, 10, or len(df_questions) for all)
num_tests = 26
# ==========================================

print("Loading dataset...")
# Load your local dataset
file_path = "dataset.csv"  # Ensure this file is in the same folder as your notebook
df_questions = pd.read_csv(file_path)

print(f"Loaded {len(df_questions)} questions from {file_path}")
if PRINT_RESULTS:
    display(df_questions.head())

# 1. Initialize the new columns in the dataframe if they don't exist
if "simple_rag_answer" not in df_questions.columns:
    df_questions["simple_rag_answer"] = ""
if "baseline_answer" not in df_questions.columns:
    df_questions["baseline_answer"] = ""

# Adjust num_tests just in case the dataset is smaller than requested
num_tests = min(num_tests, len(df_questions))
print(f"\nEvaluating {num_tests} questions. This may take a few minutes...\n")

for index, row in df_questions.head(num_tests).iterrows():
    question = row["Question"]
    expected = row["Correct Answer"]

    # 2. Get answers from both systems for comparison
    baseline_model_answer = ask_baseline(question)
    rag_model_answer = ask_rag(question)

    # 3. Save the answers into the dataframe
    df_questions.at[index, "baseline_answer"] = baseline_model_answer
    df_questions.at[index, "simple_rag_answer"] = rag_model_answer

    # 4. Pretty print the results for real-time monitoring
    if PRINT_RESULTS:
        output = f"""
### Question {index + 1}: {question}

**Expected Answer:**
> {expected}

**Baseline Model Answer (No Context):**
> {baseline_model_answer}

**RAG Model Answer (With Context):**
> {rag_model_answer}

---
"""
        display(Markdown(output))

# 5. Save the updated dataframe to your local CSV
df_questions.to_csv("dataset_with_results.csv", index=False)
print(f"✅ Successfully processed {num_tests} questions and saved to 'dataset_with_results.csv'")

Loading dataset...
Loaded 26 questions from dataset.csv


,Question,Correct Answer,Page References,Explanation
0,What type of train passes by the apartment rig...,An el train (elevated train).,26,This requires retrieving a specific environmen...
1,"According to the boy's alibi, where did he cla...",He claimed to be at the movies.,25,Retrieval of a specific plot point established...
2,What does the sign on the first door of the co...,Court of General Sessions. Part I,3,Fact explicitly stated in the script in the de...
3,What pattern does Juror #3 draw on the same sh...,tic-tac-toe pattern,64,Fact explicitly stated in the script. The info...
4,Which Juror says that he's lived in a slum all...,Juror #5,31,Fact explicitly stated in the script on page 31


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Evaluating 26 questions. This may take a few minutes...

Question: What type of train passes by the apartment right outside the window where the murder took place?
Generating baseline answer...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Question: How old is Juror #3 and what is his occupation?
Generating baseline answer...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How old is Juror #3 and what is his occupation?
Retrieving context...



### Question 26: How old is Juror #3 and what is his occupation?

**Expected Answer:**
> 40 years old. Head of a messenger service.

**Baseline Model Answer (No Context):**
> In the 1957 film *12 Angry Men*, Juror #3 is portrayed as a man in his 50s. He is approximately 50 years old. His occupation is a **retired factory worker**.

Juror #3 is known for being a strict, rigid, and initially dismissive juror who is deeply committed to the idea that the defendant is guilty. He is also shown to be somewhat prejudiced and resistant to change, which makes his eventual shift in perspective a key part of the film's theme of rational discussion and reevaluation of evidence.

**RAG Model Answer (With Context):**
> Juror #3 is 40 years old and works as the head of a messenger service.

---


✅ Successfully processed 26 questions and saved to 'dataset_with_results.csv'
